In [1]:
import os
import re
import subprocess
import json
import dill
import config
from pathlib import Path
from collections import defaultdict
from TexSoup import TexSoup
from pylatexenc.latexwalker import LatexWalker, LatexEnvironmentNode

In [2]:
DATA_FOLDER_PATH = config.DATA_FOLDER_PATH

In [20]:
ROOT_RE = re.compile(r'^\s*%\s*!TeX\s+root\s*=\s*(.+?)\s*$', re.IGNORECASE | re.MULTILINE)
DOCCLASS_RE = re.compile(r'\\documentclass(?:\[[^\]]*\])?\{[^}]+\}')
BEGIN_DOC_RE = re.compile(r'\\begin\{document\}')

COMMON_MAIN_NAMES = [
    'acl_latex.tex',
    'main.tex',
]

def detect_main_tex(project_root):
    root = Path(project_root).resolve()
    tex_files = sorted(p.resolve() for p in root.rglob("*.tex"))

    for tex_file in tex_files:
        if tex_file.name in COMMON_MAIN_NAMES:
            return tex_file
    
    main_tex_file = None
    for tex_file in tex_files:
        if tex_file.name == 'flat_main.tex':
            continue

        with open(tex_file, 'r') as fp:
            latex = fp.read()
        matches = ROOT_RE.search(latex)
        has_docclass = DOCCLASS_RE.search(latex)
        has_begin_doc = BEGIN_DOC_RE.search(latex)

        if matches:
            main_tex_file = root / matches.group(1).strip()
            return main_tex_file
        elif has_docclass and has_begin_doc:
            return tex_file
    
    return main_tex_file

In [22]:
for project in os.listdir(DATA_FOLDER_PATH):
    main_tex_file = detect_main_tex(DATA_FOLDER_PATH / project)

    if main_tex_file is None:
        print(project)
        print(main_tex_file)

.DS_Store
None
